In [1]:
# ============================================================
# Notebook    : 07_case_c_fremtpl2_eda.ipynb
# Description : Case C — freMTPL2freq EDA & preprocessing
#               - Cross-sectional (like Case B — each row is one
#                 policy-year observation; freMTPL2's IDpol CAN
#                 repeat across years in the source data, but we
#                 treat rows as independent here since there is
#                 no reliable multi-year sequence structure
#                 comparable to Case A's fremotor2freq9907b — see
#                 CHECK 2 below for the actual repeat-rate check)
#               - Label: ClaimNb >= 1 (binarized, same convention
#                 as Case A's NbClaim >= 1, for cross-Case
#                 comparability)
#               - Splits features into:
#                   C1 (static only, no BonusMalus)
#                   C2 (C1 + BonusMalus)
# ============================================================


# ============================================================
# 1. Common imports
# ============================================================
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)


# ============================================================
# 2. Load (from notebook 00 output)
# ============================================================
df = pd.read_csv("data/freMTPL2freq.csv")
print(f"[CHECK 1] Shape: {df.shape}")
print(df.head())
print(f"\n[CHECK 1] dtypes:")
print(df.dtypes)


# ============================================================
# 3. CRITICAL CHECK — does IDpol repeat across rows?
#    This determines whether freMTPL2 could ever support a
#    longitudinal treatment (it's included in this framework
#    specifically AS a cross-sectional contrast case, but we
#    verify that assumption rather than asserting it blindly)
# ============================================================
idpol_counts = df["IDpol"].value_counts()
print(f"\n[CHECK 2] Unique IDpol: {df['IDpol'].nunique():,} out of {len(df):,} rows")
print(f"[CHECK 2] IDpol repeat distribution:")
print(idpol_counts.value_counts().sort_index())
print(f"\n[CHECK 2] If IDpol never repeats (all counts == 1), this confirms")
print(f"           freMTPL2 is genuinely cross-sectional in this extract —")
print(f"           consistent with Case C's design role in the framework.")


# ============================================================
# 4. Missing value check
# ============================================================
print(f"\n[CHECK 3] Missing values per column:")
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
print(missing_summary[missing_summary["missing_count"] > 0].to_string())
if missing_summary["missing_count"].sum() == 0:
    print("  (no missing values found)")


# ============================================================
# 5. Label — binarize ClaimNb, SAME convention as Case A
#    (NbClaim >= 1), for direct cross-Case comparability
# ============================================================
print(f"\n[CHECK 4] ClaimNb distribution:")
print(df["ClaimNb"].value_counts().sort_index())

df["Label"] = (df["ClaimNb"] >= 1).astype(int)
print(f"\n[CHECK 4] Binarized Label positive rate: {df['Label'].mean()*100:.2f}%")
print(f"[CHECK 4] (Case A was 12.67-12.79%, Case B was 31.33% — "
      f"for reference)")


# ============================================================
# 6. Categorical column inventory
# ============================================================
cat_candidates = df.select_dtypes(include=["object"]).columns.tolist()
print(f"\n[CHECK 5] Object-dtype columns: {cat_candidates}")
for col in cat_candidates:
    print(f"\n  {col} — {df[col].nunique()} unique values")
    if df[col].nunique() <= 15:
        print(f"    {df[col].value_counts().to_dict()}")
    else:
        print(f"    (too many to print — top 10):")
        print(f"    {df[col].value_counts().head(10).to_dict()}")


# ============================================================
# 7. Feature grouping — C1 (static, no BonusMalus) vs
#    C2 (C1 + BonusMalus)
#    IDpol excluded (identifier). Exposure is analogous to
#    Case A's Expo — kept as a static feature in BOTH C1/C2,
#    same role as Expo played in Case A's ①②③c.
# ============================================================
STATIC_COLS = [
    "Exposure", "VehPower", "VehAge", "DrivAge",
    "VehBrand", "VehGas", "Area", "Density", "Region",
]
BONUSMALUS_COL = ["BonusMalus"]
EXCLUDED_COLS  = ["IDpol", "ClaimNb"]  # ClaimNb excluded post-binarization
LABEL_COL      = "Label"

print(f"\n[CHECK 6] Feature grouping:")
print(f"  C1 (static, no BonusMalus) : {len(STATIC_COLS)} cols — {STATIC_COLS}")
print(f"  C2 additional (BonusMalus) : {BONUSMALUS_COL}")
print(f"  Excluded                   : {EXCLUDED_COLS}")

all_expected = set(STATIC_COLS + BONUSMALUS_COL + EXCLUDED_COLS + [LABEL_COL])
all_actual   = set(df.columns)
unaccounted  = all_actual - all_expected
if unaccounted:
    print(f"\n  !! WARNING: columns present but not categorized: {unaccounted}")
else:
    print(f"\n  All {len(df.columns)} columns accounted for.")


# ============================================================
# 8. Save
# ============================================================
df.to_csv("data/fremtpl2_preprocessed.csv", index=False)
print(f"\nSaved: data/fremtpl2_preprocessed.csv")


# ============================================================
# 9. Summary
# ============================================================
print("\n===== Case C Preprocessing Summary =====")
print(f"Total rows              : {len(df):,}")
print(f"Positive rate (Label)   : {df['Label'].mean()*100:.2f}%")
print(f"C1 feature count        : {len(STATIC_COLS)}")
print(f"C2 feature count        : {len(STATIC_COLS) + len(BONUSMALUS_COL)}")
print(f"Output file             : data/fremtpl2_preprocessed.csv")
print("===========================================")

[CHECK 1] Shape: (678013, 12)
   IDpol  ClaimNb  Exposure  VehPower  VehAge  DrivAge  BonusMalus VehBrand  \
0      1        1      0.10         5       0       55          50      B12   
1      3        1      0.77         5       0       55          50      B12   
2      5        1      0.75         6       2       52          50      B12   
3     10        1      0.09         7       0       46          50      B12   
4     11        1      0.84         7       0       46          50      B12   

    VehGas Area  Density       Region  
0  Regular    D     1217  Rhone-Alpes  
1  Regular    D     1217  Rhone-Alpes  
2   Diesel    B       54     Picardie  
3   Diesel    B       76    Aquitaine  
4   Diesel    B       76    Aquitaine  

[CHECK 1] dtypes:
IDpol           int64
ClaimNb         int64
Exposure      float64
VehPower        int64
VehAge          int64
DrivAge         int64
BonusMalus      int64
VehBrand       object
VehGas         object
Area           object
Density         